# Simple RAG Implementation

In this notebook we will build a simple RAG application based on a structured CSV file with wine rating.

Steps:
- Load the dataset
- Encode a column using vector embedding
- Retrieve some of the rows based on a query using semantic similarity
- Generate a reply to the user's query based on the retrieved rows.

#### Visual Improvements
We will use `rich library` and `rich-theme-manager` to make the output more readable, and supress warning messages.

In [1]:
from rich.console import Console
from rich.style import Style
import pathlib
from rich_theme_manager import Theme, ThemeManager

THEMES = [
    Theme(
        name="dark",
        description="Dark mode theme",
        tags=["dark"],
        styles={
            "repr.own": Style(color="#e87d3e", bold=True),      # Class names
            "repr.tag_name": "dim cyan",                        # Adjust tag names 
            "repr.call": "bright_yellow",                       # Function calls and other symbols
            "repr.str": "bright_green",                         # String representation
            "repr.number": "bright_red",                        # Numbers
            "repr.none": "dim white",                           # None
            "repr.attrib_name": Style(color="#e87d3e", bold=True),    # Attribute names
            "repr.attrib_value": "bright_blue",                 # Attribute values
            "default": "bright_white on black"                  # Default text and background
        },
    ),
    Theme(
        name="light",
        description="Light mode theme",
        styles={
            "repr.own": Style(color="#22863a", bold=True),          # Class names
            "repr.tag_name": Style(color="#00bfff", bold=True),     # Adjust tag names 
            "repr.call": Style(color="#ffff00", bold=True),         # Function calls and other symbols
            "repr.str": Style(color="#008080", bold=True),          # String representation
            "repr.number": Style(color="#ff6347", bold=True),       # Numbers
            "repr.none": Style(color="#808080", bold=True),         # None
            "repr.attrib_name": Style(color="#ffff00", bold=True),  # Attribute names
            "repr.attrib_value": Style(color="#008080", bold=True), # Attribute values
            "default": Style(color="#000000", bgcolor="#ffffff"),   # Default text and background
        },
    ),
]

theme_dir = pathlib.Path("themes").expanduser()
theme_dir.expanduser().mkdir(parents=True, exist_ok=True)

theme_manager = ThemeManager(theme_dir=theme_dir, themes=THEMES)
theme_manager.list_themes()

dark = theme_manager.get("dark")
theme_manager.preview_theme(dark)

 Theme  Description       Tags  Path               
 dark   Dark mode theme   dark  themes/dark.theme  
 light  Light mode theme        themes/light.theme

                                      Theme: dark - themes/dark.theme                                      
┌───────────────────┬───────────────┬───────┬─────────┬─────────┬────────────────┬────────────────────────┐
│ style             │ color         │ color │ bgcolor │ bgcolor │ attributes     │ example                │
├───────────────────┼───────────────┼───────┼─────────┼─────────┼────────────────┼────────────────────────┤
│ default           │ bright_white  │ █████ │ black   │ █████   │ -------------- │ The quick brown fox... │
├───────────────────┼───────────────┼───────┼─────────┼─────────┼────────────────┼────────────────────────┤
│ repr.attrib_name  │ #e87d3e       │ █████ │ None    │         │ b------------- │ The quick brown fox... │
├───────────────────┼───────────────┼───────┼─────────┼─────────┼────────────────┼────────────────────────┤
│ repr.attrib_value │ bright_blue   │ █████ │ None    │         │ -------------- │ The quick brown fox... │
├───────────────────┼───────────────┼───────┼─────────┼─────────┼────────────────┼────────────────────────┤
│ repr.call         │ bright_yellow │ █████ │ None    │         │ -------------- │ The quick brown fox... │
├───────────────────┼───────────────┼───────┼─────────┼─────────┼────────────────┼────────────────────────┤
│ repr.none         │ white         │ █████ │ None    │         │ -d------------ │ The quick brown fox... │
├───────────────────┼───────────────┼───────┼─────────┼─────────┼────────────────┼────────────────────────┤
│ repr.number       │ bright_red    │ █████ │ None    │         │ -------------- │ The quick brown fox... │
├───────────────────┼───────────────┼───────┼─────────┼─────────┼────────────────┼────────────────────────┤
│ repr.own          │ #e87d3e       │ █████ │ None    │         │ b------------- │ The quick brown fox... │
├───────────────────┼───────────────┼───────┼─────────┼─────────┼────────────────┼────────────────────────┤
│ repr.str          │ bright_green  │ █████ │ None    │         │ -------------- │ The quick brown fox... │
├───────────────────┼───────────────┼───────┼─────────┼─────────┼────────────────┼────────────────────────┤
│ repr.tag_name     │ cyan          │ █████ │ None    │         │ -d------------ │ The quick brown fox... │
└───────────────────┴───────────────┴───────┴─────────┴─────────┴────────────────┴────────────────────────┘
┌─ attributes legend ──────────────────────────────────────────────────────────────────┐
│  b: bold, d: dim, i: italic, u: underline, U: double underline, B: blink, 2: blink2  │
│  r: reverse, c: conceal, s: strike, f: frame, e: encircle, o: overline, L: Link      │
└──────────────────────────────────────────────────────────────────────────────────────┘

In [2]:
from rich.console import Console

dark = theme_manager.get("dark")

# Create a console with the dark theme
console = Console(theme=dark)

In [3]:
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

## Step 1: Load the dataset

In [4]:
import pandas as pd
data = pd.read_csv("data/top_rated_wines.csv").query('variety.notna()').reset_index(drop=True).to_dict('records')
console.print(data[:2])

[
    {
        'name': '3 Rings Reserve Shiraz 2004',
        'region': 'Barossa Valley, Barossa, South Australia, Australia',
        'variety': 'Red Wine',
        'rating': 96.0,
        'notes': 'Vintage Comments : Classic Barossa vintage conditions. An average wet Spring followed by extreme 
heat in early February. Occasional rainfall events kept the vines in good balance up to harvest in late March 2004.
Very good quality coupled with good average yields. More than 30 months in wood followed by six months tank 
maturation of the blend prior to bottling, July 2007. '
    },
    {
        'name': 'Abreu Vineyards Cappella 2007',
        'region': 'Napa Valley, California',
        'variety': 'Red Wine',
        'rating': 96.0,
        'notes': 'Cappella is a proprietary blend of two clones of Cabernet Sauvignon with Cabernet Franc, Petit 
Verdot and Merlot. The gravelly soil at Cappella produces fruit that is very elegant in structure. The resulting 
wine exhibits beautiful purity of fruit with fine grained and lengthy tannins. '
    }
]

## Encode using vector embedding

We will use:
- open source vector databases: `Qdrant`  
- embedding encoder and text transformer libraries: `SentenceTransformer`


In [5]:
from qdrant_client import models, QdrantClient
from sentence_transformers import SentenceTransformer

# Create the vector database client
qdrant = QdrantClient(":memory:") # Create in memory Qdrant instance

# Create the embedding encoder
encoder = SentenceTransformer("all-MiniLM-L6-v2") # Model to create embeddings for the text data

In [12]:
# Create a collection to store the wine rating data
collection_name = "top_wines"

qdrant.recreate_collection(
    collection_name = collection_name,
    vectors_config = models.VectorParams(
        size = encoder.get_sentence_embedding_dimension(), # Vector size is defined by used model
        distance = models.Distance.COSINE # Use cosine distance for similarity search
    )
)

True

## Loading the data into the vector database
We will use the vector collection that we created above, to go over all the notes column of the wine dataset, and encode it into embedding vector, and store it in the vector database.
The indexing of the data to allow quick retrieval is running in the background as we load it.


In [16]:
qdrant.upload_points(
    collection_name = collection_name,
    points = [
        models.PointStruct(
            id = idx,
            vector = encoder.encode(doc["notes"]).tolist(),
            payload = doc
        ) for idx, doc in enumerate(data)
    ]
)

In [17]:
console.print(qdrant.get_collection(collection_name=collection_name))

CollectionInfo(
    status=<CollectionStatus.GREEN: 'green'>,
    optimizer_status=<OptimizersStatusOneOf.OK: 'ok'>,
    warnings=None,
    indexed_vectors_count=0,
    points_count=1347,
    segments_count=1,
    config=CollectionConfig(
        params=CollectionParams(
            vectors=VectorParams(
                size=384,
                distance=<Distance.COSINE: 'Cosine'>,
                hnsw_config=None,
                quantization_config=None,
                on_disk=None,
                datatype=None,
                multivector_config=None
            ),
            shard_number=None,
            sharding_method=None,
            replication_factor=None,
            write_consistency_factor=None,
            read_fan_out_factor=None,
            read_fan_out_delay_ms=None,
            on_disk_payload=None,
            sparse_vectors=None
        ),
        hnsw_config=HnswConfig(
            m=16,
            ef_construct=100,
            full_scan_threshold=10000,
            max_indexing_threads=0,
            on_disk=None,
            payload_m=None,
            inline_storage=None
        ),
        optimizer_config=OptimizersConfig(
            deleted_threshold=0.2,
            vacuum_min_vector_number=1000,
            default_segment_number=0,
            max_segment_size=None,
            memmap_threshold=None,
            indexing_threshold=20000,
            flush_interval_sec=5,
            max_optimization_threads=1,
            prevent_unoptimized=None
        ),
        wal_config=WalConfig(wal_capacity_mb=32, wal_segments_ahead=0, wal_retain_closed=1),
        quantization_config=None,
        strict_mode_config=None,
        metadata=None
    ),
    payload_schema={},
    update_queue=None
)

We will see a few sample wines stored in the database, along with their vectors.

In [20]:
records, next_offset = qdrant.scroll(
    collection_name=collection_name,
    limit=5,
    with_payload=True,
    with_vectors=True   # set False if you just want payload, not the raw numbers
)

for record in records:
    console.print(record.id, record.payload)
    console.print(record.vector[:3])  # first 10 numbers of the 384-dim vector


0
{
    'name': '3 Rings Reserve Shiraz 2004',
    'region': 'Barossa Valley, Barossa, South Australia, Australia',
    'variety': 'Red Wine',
    'rating': 96.0,
    'notes': 'Vintage Comments : Classic Barossa vintage conditions. An average wet Spring followed by extreme heat
in early February. Occasional rainfall events kept the vines in good balance up to harvest in late March 2004. Very
good quality coupled with good average yields. More than 30 months in wood followed by six months tank maturation 
of the blend prior to bottling, July 2007. '
}

[-0.052004002034664154, 0.021448031067848206, 0.04042334854602814]

1
{
    'name': 'Abreu Vineyards Cappella 2007',
    'region': 'Napa Valley, California',
    'variety': 'Red Wine',
    'rating': 96.0,
    'notes': 'Cappella is a proprietary blend of two clones of Cabernet Sauvignon with Cabernet Franc, Petit Verdot
and Merlot. The gravelly soil at Cappella produces fruit that is very elegant in structure. The resulting wine 
exhibits beautiful purity of fruit with fine grained and lengthy tannins. '
}

[-0.030425408855080605, -0.05807521566748619, -0.10527676343917847]

2
{
    'name': 'Abreu Vineyards Cappella 2010',
    'region': 'Napa Valley, California',
    'variety': 'Red Wine',
    'rating': 98.0,
    'notes': "Cappella is one of the oldest vineyard sites in St. Helena. Six acres that sit alongside a Catholic 
cemetery on the west side of town, it was first planted in 1869. In the 1980s the church asked David to tear out 
the old vines, then he watched as the land lay fallow for close to two decades. When he finally got the chance to 
replant, he jumped. He'd tasted fruit from Cappella in the 70s. He knew what kind of wine it could make. But that 
first replant was ill-fated thanks to diseased rootstock, and once again he was ripping out vines. “It took us six 
years before we had a crop. We could have ignored it, pulled the vines out one by one as they collapsed. But then 
we'd have all these different ripening patterns, which would impact consistency. It was an easy decision.”"
}

[0.028186919167637825, 0.03515014424920082, -0.06099622696638107]

3
{
    'name': 'Abreu Vineyards Howell Mountain 2008',
    'region': 'Howell Mountain, Napa Valley, California',
    'variety': 'Red Wine',
    'rating': 96.0,
    'notes': "When David purchased this Howell Mountain property in 2000 it came with an unexpected perk: first 
growth redwood stakes dating back over a century. Relics of an earlier era of agriculture. “When the college owned 
this site they'd burn all the underbrush, including the stakes, to keep it clean. When I came in we found them and 
set them all aside,” he says. At about 2000 feet elevation, Las Posadas sits above the fog line, surrounded by a 
protected forest of fir and pine. Red Aiken soils are layered over white tufa, and the rocks that littered the site
before it was planted now form walls defining the property. The redwood stakes—collected, stacked, preserved—await 
their next life."
}

[0.03692454844713211, 0.07839453220367432, 0.005371676757931709]

4
{
    'name': 'Abreu Vineyards Howell Mountain 2009',
    'region': 'Howell Mountain, Napa Valley, California',
    'variety': 'Red Wine',
    'rating': 98.0,
    'notes': 'As a set of wines, it is hard to surpass the four cuvees from the estate vineyards of David Abreu. As
I have written many times in the past, all of these wines are truly world-class efforts that stand alongside 
proprietary red wines made from Bordeaux varietals from any appellation in the world. '
}

[0.02736690267920494, -0.054024532437324524, -0.0380127914249897]

## Retrieve semantically relevant data based on user's query
Once the data is loaded into the vector database and the indexing process is done, we can start using our simple RAG system.

In [49]:
user_prompt = "Suggest me an amazing Shiraz wine from Australia"

### Encoding the user's query
We will use the same encoder that we used to encode the document data to encode the query of the user. This will search results based on semantic similarity.

In [50]:
query_vector = encoder.encode(user_prompt).tolist()

### Search similar rows
We can now take the embedding encoding of the user's query and use it to find similar rows in the vector database.

In [51]:
# Search time for awesome wines
hits = qdrant.query_points(
    collection_name = collection_name,
    query = query_vector,
    limit = 5,
    with_vectors=False
).points

console.print(hits[0])

ScoredPoint(
    id=794,
    version=0,
    score=0.6670446477289006,
    payload={
        'name': 'De Lisio Shiraz 2004',
        'region': 'McLaren Vale, South Australia, Australia',
        'variety': 'Red Wine',
        'rating': 96.0,
        'notes': '"The profound 2004 Shiraz was cropped at a measly .5 tons of fruit per acre, and aged almost 
entirely in new French oak. It is a killer wine in a killer line-up from De Lisio in 2004. Dense blue/purple to the
rim, this highly extracted (but not overly extracted) effort reveals notes of crushed rocks, blueberries, 
blackberries, camphor, lead pencil shavings, and spicy oak. Boasting great purity, a full-bodied, opulent texture, 
huge richness, but no sense of pruniness or flabbiness given its precision and refreshing structure, this is a 
well-balanced, potentially complex McLaren Vale blockbuster. It should drink well for 15+ years." - Wine Advocate'
    },
    vector=None,
    shard_key=None,
    order_value=None
)

In [52]:
from rich.console import Console
from rich.text import Text
from rich.table import Table

table = Table(title="Retrieval Results", show_lines = True)

table.add_column("Name", style = "yellow")
table.add_column("Region", style = "bright_red")
table.add_column("Variety", style = "green")
table.add_column("Rating", style = "#a6accd")
table.add_column("Notes", style = "#89ddff")
table.add_column("Score", style = "violet")

for hit in hits:
    table.add_row(
        hit.payload["name"],
        hit.payload["region"],
        hit.payload["variety"],
        str(hit.payload["rating"]),
        f'{hit.payload["notes"][:50]}...',
        f'{hit.score:.4f}'
    )
    
console.print(table)

                                                 Retrieval Results                                                 
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Name                      ┃ Region                     ┃ Variety  ┃ Rating ┃ Notes                     ┃ Score  ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ De Lisio Shiraz 2004      │ McLaren Vale, South        │ Red Wine │ 96.0   │ "The profound 2004 Shiraz │ 0.6670 │
│                           │ Australia, Australia       │          │        │ was cropped at a measly   │        │
│                           │                            │          │        │ ...                       │        │
├───────────────────────────┼────────────────────────────┼──────────┼────────┼───────────────────────────┼────────┤
│ De Lisio The Catalyst     │ McLaren Vale, South        │ Red Wine │ 96.0   │ "A blockbuster in the     │ 0.6392 │
│ Shiraz/Grenache 2004      │ Australia, Australia       │          │        │ making, the 2004          │        │
│                           │                            │          │        │ Shiraz/Gren...            │        │
├───────────────────────────┼────────────────────────────┼──────────┼────────┼───────────────────────────┼────────┤
│ Chris Ringland Shiraz     │ Barossa Valley, Barossa,   │ Red Wine │ 99.0   │ We only get one bottle    │ 0.6303 │
│ 1998                      │ South Australia, Australia │          │        │ each year of the 1200     │        │
│                           │                            │          │        │ made....                  │        │
├───────────────────────────┼────────────────────────────┼──────────┼────────┼───────────────────────────┼────────┤
│ By Farr Shiraz 2014       │ Geelong, Victoria,         │ Red Wine │ 96.0   │ A powerful nose, with the │ 0.6280 │
│                           │ Australia                  │          │        │ depth and complexity of   │        │
│                           │                            │          │        │ ...                       │        │
├───────────────────────────┼────────────────────────────┼──────────┼────────┼───────────────────────────┼────────┤
│ Greenock Creek Alices     │ Barossa Valley, Barossa,   │ Red Wine │ 96.0   │ Rich and fleshy, with     │ 0.6203 │
│ Shiraz 2002               │ South Australia, Australia │          │        │ pretty coffee, plum, wild │        │
│                           │                            │          │        │ be...                     │        │
└───────────────────────────┴────────────────────────────┴──────────┴────────┴───────────────────────────┴────────┘

## Augment the prompt to the LLM with retrieved data
We will simply take the top 5 results and use them as in the prompt to the generation LLM.

## Generate reply to the user's query
We will use genAI LMM from OpenAI.

In [53]:
from dotenv import load_dotenv
load_dotenv()

True

#### First let's try without retieval
We can ask the LLM to recommend based on the user prompt.

In [ ]:
# Now time to connect to the LLM
from openai import OpenAI
from rich.panel import Panel

client = OpenAI()
completion = client.chat.completions.create(
    model = "gpt-4o",
    messages = [
        {
            "role": "system",
            "content": "You are a chatbot, a wine specialist. Your top priority is to help guide users to select amazing wine and guide them with their requests."
        },
        {
            "role": "user",
            "content": user_prompt
        },
        {
            "role": "assistant",
            "content": f"Here are some amazing wines that I found for you:\n\n"
        }
    ]
)

response_text = Text(completion.choices[0].message.content)

styled_panel = Panel(
    response_text,
    title="Wine Recommendation without Retrieval",
    expand=False,
    border_style="bold green",
    padding=(1, 1)
)

console.print(styled_panel)

╭───────────────────────────────────── Wine Recommendation without Retrieval ─────────────────────────────────────╮
│                                                                                                                 │
│ Here are some amazing Shiraz wines from Australia that you might enjoy:                                         │
│                                                                                                                 │
│ 1. **Penfolds Grange** - This iconic wine is often considered Australia's flagship Shiraz. It is rich,          │
│ full-bodied, and known for its complexity, with flavors of dark fruit, chocolate, and spice. It has great aging │
│ potential.                                                                                                      │
│                                                                                                                 │
│ 2. **Clonakilla Shiraz Viognier** - This wine is a unique blend of Shiraz and a small amount of Viognier, which │
│ adds aromatic qualities. It showcases a beautiful balance of fruit and floral notes, with a silky texture.      │
│                                                                                                                 │
│ 3. **Henschke Hill of Grace** - Another top-tier Shiraz, this wine is sourced from a single vineyard in the     │
│ Eden Valley. It offers intense flavors of blackberry, plum, and a hint of spice and is well-structured with a   │
│ long finish.                                                                                                    │
│                                                                                                                 │
│ 4. **d'Arenberg The Dead Arm Shiraz** - Known for its full-bodied character and rich flavor profile, The Dead   │
│ Arm features notes of dark chocolate, plum, and earthy undertones. It's complex and has aging potential.        │
│                                                                                                                 │
│ 5. **Yalumba The Signature** - This is a classic example of Australian Shiraz, combining ripe fruit flavors     │
│ with spicy complexity. It’s well-balanced and approachable, making it a great choice for both new and seasoned  │
│ wine drinkers.                                                                                                  │
│                                                                                                                 │
│ 6. **Torbreck The Struie** - A blend of Shiraz from Barossa Valley and Eden Valley, this wine showcases both    │
│ ripe fruit and peppery spice, with a lush mouthfeel and elegant tannins.                                        │
│                                                                                                                 │
│ Each of these wines has its own unique character, so your choice might depend on your personal taste            │
│ preferences such as fruitiness, spiciness, or complexity. Enjoy your tasting!                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

#### Now, add Retrieval Results
The recommendation sounds great, however we don't have this wine in our inventory and menu. Moreover, new wines may be newly available that were not part of the pre-training of the LLM.

We will run the same query with the Retrieval Results and get better recommendattions for our business needs.

In [55]:
# Define a variable to hold the search results
search_results = [hit.payload for hit in hits]

In [56]:
console.print(search_results)

[
    {
        'name': 'De Lisio Shiraz 2004',
        'region': 'McLaren Vale, South Australia, Australia',
        'variety': 'Red Wine',
        'rating': 96.0,
        'notes': '"The profound 2004 Shiraz was cropped at a measly .5 tons of fruit per acre, and aged almost 
entirely in new French oak. It is a killer wine in a killer line-up from De Lisio in 2004. Dense blue/purple to the
rim, this highly extracted (but not overly extracted) effort reveals notes of crushed rocks, blueberries, 
blackberries, camphor, lead pencil shavings, and spicy oak. Boasting great purity, a full-bodied, opulent texture, 
huge richness, but no sense of pruniness or flabbiness given its precision and refreshing structure, this is a 
well-balanced, potentially complex McLaren Vale blockbuster. It should drink well for 15+ years." - Wine Advocate'
    },
    {
        'name': 'De Lisio The Catalyst Shiraz/Grenache 2004',
        'region': 'McLaren Vale, South Australia, Australia',
        'variety': 'Red Wine',
        'rating': 96.0,
        'notes': '"A blockbuster in the making, the 2004 Shiraz/Grenache The Catalyst (80% Shiraz and 20% Grenache 
aged in primarily neutral French wood) boasts fabulous aromas of flowers, lead pencil shavings, blackberries, 
cassis, and subtle wood. This full-bodied, intensely packed and stacked effort possesses huge fruit extract, 
wonderful, mouth-coating glycerin, and a purity as well as seamlessness that must be tasted to be believed. Drink 
this undeniably profound Australian red over the next 10-15 years." - Wine Advocate'
    },
    {
        'name': 'Chris Ringland Shiraz 1998',
        'region': 'Barossa Valley, Barossa, South Australia, Australia',
        'variety': 'Red Wine',
        'rating': 99.0,
        'notes': 'We only get one bottle each year of the 1200 made.  Enjoy!A compelling effort from Chris Ringland
(the name Three Rivers was dropped because of trademark litigation), this is his classic 1,200 bottle Shiraz cuvee 
from a 90-year old Barossa vineyard that yielded only one ton of fruit per acre. Aged in 300 liter French hogsheads
(100% new) for 42 months prior to bottling, this is a prodigious offering from an exceptional vintage for South 
Australia. An inky purple color is followed by a gorgeous nose of minerals, blackberry liqueur, barbecue spices, 
sweet licorice, and a hint of white flowers. Remarkably concentrated, full-bodied, and unctuously-textured, its 
purity, definition, and compelling palate persistence as well as complexity are awesome. This is a magical wine 
from a winemaker who is a master craftsman. Anticipated maturity: 2008-2020+.'
    },
    {
        'name': 'By Farr Shiraz 2014',
        'region': 'Geelong, Victoria, Australia',
        'variety': 'Red Wine',
        'rating': 96.0,
        'notes': 'A powerful nose, with the depth and complexity of cool-climate shiraz. This wine is spiced with 
pepper and mineral elements, leaning towards earthy. The co-fermented viognier adds a little richness to both the 
bouquet and palate, which has a very pleasant sweetness to start, followed by intense fruit and earthy long tannins
to complete the delicate structure and overall elegance of the wine.'
    },
    {
        'name': 'Greenock Creek Alices Shiraz 2002',
        'region': 'Barossa Valley, Barossa, South Australia, Australia',
        'variety': 'Red Wine',
        'rating': 96.0,
        'notes': 'Rich and fleshy, with pretty coffee, plum, wild berry and spice notes that are smooth and 
polished, long and flavorful.  An extremely limited release wine from one of Australia\'s "Cult" wineries.'
    }
]

In [58]:
completion_with_retrieval = client.chat.completions.create(
    model = "gpt-4o",
    messages = [
        {
            "role": "system",
            "content": "You are a chatbot, a wine specialist. Your top priority is to help guide users to select amazing wine and guide them with their requests."
        },
        {
            "role": "user",
            "content": user_prompt
        },
        {
            "role": "assistant",
            "content": str(search_results)
        }
    ]
)   

response_text_with_retrieval = Text(completion_with_retrieval.choices[0].message.content)

styled_panel_with_retrieval = Panel(
    response_text_with_retrieval,
    title="Wine Recommendation with Retrieval",
    expand=False,
    border_style="bold green",
    padding=(1, 1)
)

console.print(styled_panel_with_retrieval)

╭────────────────────────────────────── Wine Recommendation with Retrieval ───────────────────────────────────────╮
│                                                                                                                 │
│ Here are a few outstanding Australian Shiraz wines you might consider:                                          │
│                                                                                                                 │
│ 1. **De Lisio Shiraz 2004**                                                                                     │
│    - **Region**: McLaren Vale, South Australia                                                                  │
│    - **Rating**: 96                                                                                             │
│    - **Tasting Notes**: Offers dense blue/purple color with aromas of crushed rocks, blueberries, blackberries, │
│ and spicy oak. It has a full-bodied, opulent texture and a refreshing structure, promising complexity and       │
│ longevity.                                                                                                      │
│                                                                                                                 │
│ 2. **Chris Ringland Shiraz 1998**                                                                               │
│    - **Region**: Barossa Valley, South Australia                                                                │
│    - **Rating**: 99                                                                                             │
│    - **Tasting Notes**: This limited production wine boasts a nose of minerals, blackberry liqueur, barbecue    │
│ spices, and white flowers. It's remarkably concentrated with full-bodied richness, complexity, and a            │
│ long-lasting finish.                                                                                            │
│                                                                                                                 │
│ 3. **By Farr Shiraz 2014**                                                                                      │
│    - **Region**: Geelong, Victoria                                                                              │
│    - **Rating**: 96                                                                                             │
│    - **Tasting Notes**: This cool-climate Shiraz offers complexity with spices, minerals, and an earthy         │
│ character. It features a sweetness followed by intense fruit and long tannins, showcasing elegance and depth.   │
│                                                                                                                 │
│ 4. **Greenock Creek Alices Shiraz 2002**                                                                        │
│    - **Region**: Barossa Valley, South Australia                                                                │
│    - **Rating**: 96                                                                                             │
│    - **Tasting Notes**: Known for its rich and fleshy profile, it combines notes of coffee, plum, wild berry,   │
│ and spice with a smooth, polished texture. It's from one of Australia’s "Cult" wineries.                        │
│                                                                                                                 │
│ These wines are known for their robust and complex flavors, embodying the essence of Australian Shiraz. Enjoy   │
│ your exploration!                                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Results


### Without Retrieval

**Steps:**
1. The `user_prompt` is sent directly to `gpt-4o` as-is, with only a system message telling it to act as a wine specialist.
2. The LLM answers purely from its own training knowledge — no data from the CSV/Qdrant database is involved.

**Response:** It suggested well-known, famous wines (Penfolds Grange, Clonakilla, Henschke Hill of Grace, etc.) — general knowledge picks that a wine expert LLM would "just know," none of which are guaranteed to be in the dataset or verified against real ratings/notes.

### With Retrieval (RAG)

**Steps:**
1. `user_prompt` is encoded into a 384-dim vector using the same `SentenceTransformer` model.
2. That vector is used to search (`query_points`) the `top_wines` Qdrant collection, returning the top 5 wines whose embeddings are most similar in meaning.
3. Those retrieved wines (with their real name, region, rating, and tasting notes) are inserted into the prompt sent to `gpt-4o`, alongside the original `user_prompt`.
4. The LLM then generates its answer using this retrieved context as grounding material, rather than just its training data.

**Response:** It recommended specific wines *from the actual dataset* (De Lisio Shiraz 2004, Chris Ringland Shiraz 1998, By Farr Shiraz 2014, Greenock Creek Alices Shiraz 2002) — each with real region, rating, and tasting notes pulled directly from the CSV, not invented by the model.

### Key Difference

Without retrieval, the LLM (`gpt-4o`) answers from memory — plausible but ungrounded and unverifiable (it can't cite a rating or confirm the wine is in the inventory). With retrieval, the LLM is constrained to recommend real wines from the specific dataset, with verifiable facts (ratings, regions, notes) attached — this is the core value RAG adds: grounding generative answers in trusted, real data.


## Summary

We built a simple RAG (Retrieval-Augmented Generation) pipeline for wine recommendations. 

We took a dataset of 1,366 rated wines, converted each wine's tasting notes into a numerical vector (embedding) using a sentence-transformer model, and stored these vectors in a Qdrant vector database. 

We then took a user's wine request, converted it into a vector the same way, and searched the database for the most semantically similar wines. 

Finally, we passed those retrieved wines to an LLM (gpt-4o) to generate a natural, personalized recommendation grounded in real data from our dataset.